# 회전행렬 R 직접 추정 — target 1 rep 캘리 + source 집계

samsung1(source) → samsung2(target) IMU 축차를, **gravity alignment 베스트 세션(latpulldown)** 으로 직접 푼다.

**설계 원칙(근거 기반):**
- 캘리브레이션 제약은 **target 쪽**에만 있다 → **target = 1 rep**(beep 분절). source(samsung1)는 이미 가진 도메인이라 1 rep 으로 제한할 이유가 없다 → **source = 집계 표현(분포/레퍼런스)**.
- 정지 스냅샷(중력 1벡터)은 tilt(2DOF)만 → yaw 미결정이라 이전엔 탐색(70.5%). 1 rep 시계열엔 중력+모션이 모두 있어 **full R 을 탐색 없이 직접** 구한다.
- source 범위 **2가지 비교**: `sub2`(베스트와 동일 피험자 latpulldown 전체) vs `full`(전체 도메인 sub1~14).
- R 추정 4종: A.Procrustes(DTW) · B.Kabsch(중력+모션축) · C.PCA프레임 · D.DTW-loss최적화
- 기준선: raw / 수동 YXZ / yaw-sweep 70.5%
- ⚠️ source·target 은 **다른 사람**(sub2/sub7) → 경로 차이가 R 잔차로 섞인다(원리적 한계). 그래서 source 를 집계로 안정화하는 게 핵심.

## 0. 셋업 — 회전 유틸

In [ ]:
import os, sys, importlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 폰트
from scipy.optimize import minimize

ROOT = os.path.abspath('../..')
sys.path.insert(0, ROOT)
import eval_utils as U
importlib.reload(U)

IMU = U.IMU_COLS  # ['triceps_X','triceps_Y','triceps_Z']
N = 200           # 리샘플 길이

# 기준 세션 (gravity alignment 베스트: latpulldown)
LP_SRC_FILE = 'sub2_latpulldown_plot_and_store_rep_2.143.csv'                # samsung1 / sub2 (viz용 1사이클)
LP_TGT_FILE = 'Data_NINA11_2025.09.16_10.42.27_sub7_latpulldown_1_59_6.csv'  # samsung2 / sub7 (캘리 1 rep)

def unit(v):
    v = np.asarray(v, float); return v / (np.linalg.norm(v) + 1e-12)
def unit_rows(M):
    M = np.asarray(M, float); return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
def angle_deg(a, b):
    return float(np.degrees(np.arccos(np.clip(np.dot(unit(a), unit(b)), -1, 1))))
def skew(v):
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def expmap(r):
    # axis-angle 벡터 r(3,) -> 회전행렬 (Rodrigues), |r|=각[rad]
    th = np.linalg.norm(r)
    if th < 1e-12: return np.eye(3)
    K = skew(r/th)
    return np.eye(3) + np.sin(th)*K + (1-np.cos(th))*(K@K)
def logmap(R):
    th = np.arccos(np.clip((np.trace(R)-1)/2, -1, 1))
    if th < 1e-8: return np.zeros(3)
    w = np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]]) / (2*np.sin(th))
    return w*th
def kabsch(src, tgt):
    # src_i ~= R @ tgt_i 인 proper rotation R (det=+1). 행벡터: src ~= tgt @ R.T
    H = np.asarray(tgt).T @ np.asarray(src)
    U_, _, Vt = np.linalg.svd(H); V = Vt.T
    d = np.sign(np.linalg.det(V @ U_.T))
    return V @ np.diag([1,1,d]) @ U_.T

print('helpers ready | 기준:', LP_SRC_FILE, '<->', LP_TGT_FILE)

## 1. 모델 / 데이터 로드

In [ ]:
model   = U.load_mm_model(os.path.join(ROOT, 'weights', 'mm_no_da_seed42_best_model.pth'))
S1      = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung1.parquet'),
                          columns=IMU + ['filename','timestamp','exercise','subject'])
dtw_ref = U.build_dtw_reference(S1, session_col='filename', time_col='timestamp')
df_tgt  = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung2_original.parquet'))
print('로드 완료 |', model.__class__.__name__, '| S1', S1.shape, '| df_tgt', df_tgt.shape)

## 2. 평가 하네스 + 기준선

`acc_dtw(R)` = R(target→source) 적용 후 **정확도(MM 모델 추론)** 와 **DTW(라벨-free 정렬도)** 반환.
EMG 전처리는 `MMEvalCache` 로 1회만(이후 IMU 변환만 반복). > ⚠️ 캐시 빌드 + 추론이라 무겁다(CPU 느림).

In [ ]:
# ★ 무거운 셀: EMG 전처리 캐시 1회 빌드
cache = U.MMEvalCache(df_tgt, model=model)
YXZ   = np.array([[0,1,0],[1,0,0],[0,0,1.]])  # 수동 X<->Y swap baseline

def target_raw_segments(df, ref_keys, min_len=20):
    segs = {}
    for ex in ref_keys:
        sub = df[df['exercise'] == ex]
        if not len(sub): continue
        sess = sorted(sub['csv_filename_l'].unique())[0]
        mat = (sub[sub['csv_filename_l'] == sess].sort_values('Index_Time')[IMU].to_numpy(np.float64))
        if len(mat) >= min_len: segs[ex] = mat
    return segs
tgt_raw = target_raw_segments(df_tgt, dtw_ref.keys())

def acc_dtw(R):
    # R(target->source) 의 (정확도%, 평균DTW)
    acc = cache.accuracy(lambda a, R=R: a @ R.T)
    per = [np.mean([U._dtw_mv(s, U._resample(U._zscore(tgt_raw[ex] @ R.T), N)) for s in src])
           for ex, src in dtw_ref.items() if ex in tgt_raw and src]
    return acc, float(np.mean(per))

RESULTS = {}
RMATS = {}
raw_acc, raw_dtw = acc_dtw(np.eye(3))
yxz_acc, yxz_dtw = acc_dtw(YXZ)
YAW_SWEEP_BEST = 70.51  # 참고: 단일중력 tilt + yaw 30deg x12 탐색(latpulldown) 베스트
RESULTS['raw (identity)'] = (raw_acc, raw_dtw)
RESULTS['수동 YXZ']        = (yxz_acc, yxz_dtw)
print(f'캐시 윈도우 {cache.n_windows}')
print(f'[baseline] raw      acc={raw_acc:5.2f}%  dtw={raw_dtw:.1f}')
print(f'[baseline] 수동 YXZ  acc={yxz_acc:5.2f}%  dtw={yxz_dtw:.1f}')
print(f'[참고]     yaw-sweep acc={YAW_SWEEP_BEST:.2f}%   <- 직접-R 목표선')

## 3. 캘리 데이터 — 무엇을 골랐나 (증거)

**target = beep 으로 자른 1 rep**, **source = 집계**(분포/레퍼런스). 먼저 무엇을 골랐는지 시각화로 확인한다.

In [ ]:
def session_imu(df, sess_col, sess, time_col):
    s = df[df[sess_col] == sess].sort_values(time_col)
    t = s[time_col].to_numpy(float); t = t - t[0]
    return t, s[IMU].to_numpy(np.float64)

# TARGET: 전체 세션 + beep onset 으로 1 rep
t_tgt, A_tgt = session_imu(df_tgt, 'csv_filename_l', LP_TGT_FILE, 'Index_Time')
beep_tgt = (df_tgt[df_tgt['csv_filename_l'] == LP_TGT_FILE].sort_values('Index_Time')['beep'].to_numpy())
onsets_idx = np.where((beep_tgt[1:] != 0) & (beep_tgt[:-1] == 0))[0] + 1
BEEP_I = 1                                  # onsets[i]~onsets[i+1] = 1 rep
i0, i1 = onsets_idx[BEEP_I], onsets_idx[BEEP_I + 1]
rep_tgt = A_tgt[i0:i1]                       # ★ 캘리에 쓰는 유일한 target 데이터(1 rep)

# SOURCE: 클립 전체 + (시각화용) 자기상관 1사이클 — 추정엔 안 씀, 집계가 대체
t_src, A_src = session_imu(S1, 'filename', LP_SRC_FILE, 'timestamp')
fs_src = 1.0 / np.median(np.diff(t_src))
def dominant_period(X, fs, pmin=0.6, pmax=8.0):
    x = X.mean(1); x = x - x.mean()
    ac = np.correlate(x, x, 'full')[len(x)-1:]
    lo, hi = int(pmin*fs), min(int(pmax*fs), len(ac)-1)
    return lo + int(np.argmax(ac[lo:hi]))
P = dominant_period(A_src, fs_src)
mag = np.linalg.norm(A_src - A_src.mean(0), axis=1)
s0 = int(np.clip(np.argmax(mag) - P//2, 0, len(A_src) - P - 1))
src_cyc = A_src[s0:s0 + P]                   # 시각화용 1사이클

zs = lambda M: U._resample(U._zscore(M), N)
print(f'TARGET 1 rep: beep{BEEP_I}~{BEEP_I+1}  t=[{t_tgt[i0]:.2f},{t_tgt[i1]:.2f}]s  ({len(rep_tgt)} samples)')
print(f'SOURCE clip {t_src[-1]:.1f}s, fs≈{fs_src:.0f}Hz | viz 1사이클 {P/fs_src:.2f}s')

### 3-1. 선택 시각화 — 이게 한 rep 이다

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 9))
# (1) source 클립 전체 + 시각화용 1사이클 강조
for k, c in enumerate(IMU): ax[0].plot(t_src, A_src[:, k], lw=.7, label=c)
ax[0].axvspan(t_src[s0], t_src[s0+P], color='orange', alpha=.3)
ax[0].set_title(f'SOURCE sub2 latpulldown 클립(15s) — 주황=자기상관 1사이클({t_src[s0]:.1f}~{t_src[s0+P]:.1f}s) · 추정엔 집계 사용')
ax[0].legend(fontsize=7, loc='upper right'); ax[0].grid(alpha=.3); ax[0].set_xlabel('t [s]')
# (2) target 세션 전체 + beep + 선택 rep 강조
for k, c in enumerate(IMU): ax[1].plot(t_tgt, A_tgt[:, k], lw=.7, label=c)
for o in onsets_idx: ax[1].axvline(t_tgt[o], color='red', ls='--', lw=1, alpha=.6)
ax[1].axvspan(t_tgt[i0], t_tgt[i1], color='orange', alpha=.3)
ax[1].set_title(f'TARGET sub7 latpulldown(39s) — 빨강=beep onset, 주황=선택 1 rep(beep{BEEP_I}~{BEEP_I+1}) ★캘리 데이터')
ax[1].legend(fontsize=7, loc='upper right'); ax[1].grid(alpha=.3); ax[1].set_xlabel('t [s]')
# (3) 정규화 오버레이 — source 1사이클(solid) vs target rep(dashed)
Zc, Zt = zs(src_cyc), zs(rep_tgt)
for k in range(3):
    ax[2].plot(Zc[:, k], lw=1.2, color=f'C{k}', label=f'src {IMU[k]}' if k==0 else None)
    ax[2].plot(Zt[:, k], lw=1.2, ls='--', color=f'C{k}')
ax[2].set_title('정규화 오버레이: solid=source 1사이클, dashed=target 1 rep (모양 차이=사람간/잡음 → source 집계로 완화)')
ax[2].grid(alpha=.3); ax[2].legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.show()

## 4. source 집계 표현 만들기 (sub2 vs full)

`target = rep_tgt`(1 rep) 고정. source 는 latpulldown 을 **집계**해서:
- `points` : 분포 방법(B 중력·모션축, C PCA)용 raw IMU 점구름
- `segs`   : 시계열 방법(A Procrustes, D DTW-loss)용 레퍼런스 세그먼트(여러 rep)

scope `sub2`=sub2 전체 / `full`=전체 도메인(피험자별 1세션).

In [ ]:
def build_src(scope, max_seg=6, cap=150000):
    sub = S1[S1.exercise == 'latpulldown']
    if scope == 'sub2':
        sub = sub[sub.subject == 'sub2']
        files = sorted(sub.filename.unique())[:max_seg]
    else:  # full: 도메인 대표로 피험자별 첫 세션
        files = []
        for subj in sorted(sub.subject.unique()):
            ff = sorted(sub[sub.subject == subj].filename.unique())
            if ff: files.append(ff[0])
        files = files[:max_seg]
    pts = sub[IMU].to_numpy(np.float64)
    step = max(1, len(pts)//cap); pts = pts[::step]
    segs = []
    for f in files:
        m = sub[sub.filename == f].sort_values('timestamp')[IMU].to_numpy(np.float64)
        if len(m) >= 20: segs.append(U._resample(m, N))
    return dict(points=pts, files=files,
                segs_z=[U._zscore(s) for s in segs],
                segs_dir=[unit_rows(s) for s in segs])

SRC = {sc: build_src(sc) for sc in ['sub2', 'full']}
for sc, d in SRC.items():
    print(f"[{sc:4s}] points={len(d['points']):6d}  segs={len(d['segs_z'])}  files={[f[:18] for f in d['files']][:3]}...")

def planarity(M):
    sv = np.linalg.svd(M - M.mean(0), compute_uv=False); return sv / sv[0]
print('target rep planarity s2/s0 =', round(planarity(rep_tgt)[2], 3), '(작을수록 평면적=yaw 약함)')

## 5. R 추정 방법 정의 (A~D)

각 함수는 `scope` 의 source 집계와 `rep_tgt` 로 R(target→source) 반환. 부호/후보 모호성은
**source 집계 세그먼트에 대한 rep-DTW**(라벨-free)로 택1 → 평가지표와 일치.

In [ ]:
def dtw_path(a, b):
    n, m = len(a), len(b)
    cost = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(-1))
    D = np.full((n+1, m+1), np.inf); D[0, 0] = 0
    for i in range(1, n+1):
        for j in range(1, m+1):
            D[i, j] = cost[i-1, j-1] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    i, j, path = n, m, []
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        s = int(np.argmin([D[i-1, j-1], D[i-1, j], D[i, j-1]]))
        i, j = [(i-1, j-1), (i-1, j), (i, j-1)][s]
    return path[::-1]

def gravity_axis(M):
    g = unit(M.mean(0))                                  # 중력(평균방향)
    pc1 = np.linalg.svd(M - M.mean(0), full_matrices=False)[2][0]
    return g, unit(pc1 - np.dot(pc1, g) * g)             # 중력 직교화한 모션 주축
def pca_frame(M):
    return np.linalg.svd(M - M.mean(0), full_matrices=False)[2]   # rows=주축(내림차순)

def rep_dtw_segs(R, scope):
    # 회전된 target rep 과 source 집계 세그먼트들의 평균 DTW (싸다; 선택·최적화용)
    zt = zs(rep_tgt @ R.T)
    return float(np.mean([U._dtw_mv(sz, zt) for sz in SRC[scope]['segs_z']]))

def m_procrustes(scope):
    segs_dir, segs_z = SRC[scope]['segs_dir'], SRC[scope]['segs_z']
    tgt_d = unit_rows(U._resample(rep_tgt, N))
    R = np.eye(3)
    for _ in range(6):                                   # ICP: 각 source seg 와 DTW대응 -> Kabsch
        zt = U._zscore(U._resample(rep_tgt, N) @ R.T)
        SI, TI = [], []
        for sd, sz in zip(segs_dir, segs_z):
            path = dtw_path(sz, zt)
            SI.append(sd[[p[0] for p in path]]); TI.append(tgt_d[[p[1] for p in path]])
        R = kabsch(np.vstack(SI), np.vstack(TI))
    return R

def m_kabsch2(scope):
    g_s, ax_s = gravity_axis(SRC[scope]['points'])
    g_t, ax_t = gravity_axis(rep_tgt)
    bR = lambda s: kabsch(np.vstack([g_s, ax_s, np.cross(g_s, ax_s)]),
                          np.vstack([g_t, s*ax_t, np.cross(g_t, s*ax_t)]))
    return min([bR(1), bR(-1)], key=lambda R: rep_dtw_segs(R, scope))

def m_pca(scope):
    Fs, Ft = pca_frame(SRC[scope]['points']), pca_frame(rep_tgt)
    best = None
    for sg in itertools.product((1,-1), repeat=3):
        R = Fs.T @ np.diag(sg) @ Ft
        if np.linalg.det(R) < 0: continue
        sc = rep_dtw_segs(R, scope)
        best = (sc, R) if best is None or sc < best[0] else best
    return best[1]

def m_dtwloss(scope):
    R0 = U.rotation_align(unit(rep_tgt.mean(0)), unit(SRC[scope]['points'].mean(0)))  # tilt 초기화
    res = minimize(lambda r: rep_dtw_segs(expmap(r), scope), logmap(R0),
                   method='Powell', options={'maxiter':250, 'xtol':1e-3, 'ftol':1e-3})
    return expmap(res.x)

METHODS = {'A. Procrustes(DTW)': m_procrustes, 'B. Kabsch(중력+모션축)': m_kabsch2,
           'C. PCA 프레임': m_pca, 'D. DTW-loss': m_dtwloss}
print('methods ready:', list(METHODS))

## 6. 실행 — 각 방법 × {sub2, full} (정확도 + 회전행렬 R 출력)

In [ ]:
for scope in ['sub2', 'full']:
    for name, fn in METHODS.items():
        R = fn(scope)
        a, d = acc_dtw(R)
        RESULTS[f'{name} [{scope}]'] = (a, d); RMATS[f'{name} [{scope}]'] = R
        print(f'{name:22s} [{scope:4s}]  acc={a:5.2f}%  dtw={d:.1f}   R(target→source, imu @ R.T):')
        print(np.round(R, 4))

## 7. 종합 비교

In [ ]:
summ = (pd.DataFrame([{'방법': k, 'accuracy(%)': round(v[0],2), 'DTW': round(v[1],1)}
                      for k, v in RESULTS.items()])
        .sort_values('accuracy(%)', ascending=False).reset_index(drop=True))
display(summ)
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tab:gray' if '[' not in m else ('tab:blue' if 'sub2' in m else 'tab:purple')
          for m in summ['방법'][::-1]]
ax.barh(summ['방법'][::-1], summ['accuracy(%)'][::-1], color=colors, alpha=.85)
ax.axvline(raw_acc, color='gray', ls='--', label=f'raw {raw_acc:.1f}%')
ax.axvline(yxz_acc, color='tab:red', ls=':', label=f'YXZ {yxz_acc:.1f}%')
ax.axvline(YAW_SWEEP_BEST, color='tab:green', ls='-.', label=f'yaw-sweep {YAW_SWEEP_BEST:.1f}%')
ax.set_xlabel('accuracy (%)'); ax.legend(fontsize=8)
ax.set_title('직접 R 추정: 방법 × source 범위(파랑=sub2, 보라=full) vs 기준선')
plt.tight_layout(); plt.show()

# ── best 방법의 회전행렬 R 출력 ──
best_name = summ.iloc[0]['방법']
if best_name in RMATS:
    print(f"★ best: {best_name}  acc={summ.iloc[0]['accuracy(%)']}%  R(target→source, imu @ R.T):")
    print(np.round(RMATS[best_name], 4))